In [3]:
# 梯度下降（数值微分版）
import numpy as np


# 计算梯度：对每个维度做中心差分

def numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x)

    for i in range(len(x)):
        tmp = x[i]  # 先保存当前值

        x[i] = tmp + h  # 只改第 i 个元素
        fxh1 = f(x)  # f(x + h)

        x[i] = tmp - h  # 只改第 i 个元素
        fxh2 = f(x)  # f(x - h)

        grad[i] = (fxh1 - fxh2) / (2 * h)  # 第 i 个偏导
        x[i] = tmp  # 还原

    return grad


# 梯度下降：沿负梯度方向更新参数

def gradient_descent(f, init_x, lr=0.01, step_num=100):
    x = init_x.copy()  # 避免修改原数组
    x_history = []

    for _ in range(step_num):
        x_history.append(x.copy())
        grad = numerical_gradient(f, x)
        x -= lr * grad  # 沿负梯度方向走一步

    return x, np.array(x_history)


## 测试例子：z = x^2 + x*y + y^2

我们用梯度下降寻找这个函数的最小值点。

解析上它的最小值在 $(0,0)$，梯度下降应该逐步逼近这个点。


In [4]:
# 测试函数：z = x^2 + x*y + y^2


def f_xy(v):
    x, y = v[0], v[1]
    return x ** 2 + x * y + y ** 2


init_x = np.array([0.6, 0.9])
result, history = gradient_descent(f_xy, init_x, lr=0.1, step_num=30)

print("最终位置:", result)
print("前 5 步:")
print(history[:5])


最终位置: [-0.00634177  0.00637558]
前 5 步:
[[0.6     0.9    ]
 [0.39    0.66   ]
 [0.246   0.489  ]
 [0.1479  0.3666 ]
 [0.08166 0.27849]]


## Sigmoid 函数（数学表达式）

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

特点：
- 输出范围在 $(0, 1)$
- 常用于二分类输出层或作为激活函数


## 交叉熵损失（cross_entropy_error）

多分类交叉熵（批量版）：

$$
L = -\frac{1}{N}\sum_{i=1}^{N}\log\big(p_{i,\text{true}}\big)
$$

说明：
- $p_{i,\text{true}}$ 是第 $i$ 个样本的“正确类别概率”
- 输入可以是 one-hot 标签，也可以是类别索引


In [ ]:
import numpy as np

# softmax：把得分转成概率

def softmax(x):
    if x.ndim == 2:
        x = x - np.max(x, axis=1, keepdims=True)
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


# 交叉熵损失：支持 one-hot 或类别索引

def cross_entropy_error(y, t):
    if y.ndim == 1:
        y = y.reshape(1, -1)
        t = t.reshape(1, -1)

    if t.size == y.size:
        t = np.argmax(t, axis=1)

    batch_size = y.shape[0]
    delta = 1e-7  # 防止 log(0)

    return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size


In [ ]:
## 手写数字识别（简化两层网络）
## sigmoid(x) = 1 / (1 + e^(-x))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']

        a1 = np.dot(x, W1) + b1
        # 激活函数
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)  # 输出概率
        return y

    def loss(self, x, t):
        y = self.predict(x)
        return cross_entropy_error(y, t)
